# ADM denoising without training: noise level from the image, and the mean of K posterior samples

Two inference-only changes to the frozen ADM 256×256 denoiser of `jepa-guided-diffusion`, on its 52 evaluation cases (same inputs and seeds, checked against its `scores.csv`):

1. **Start step from the noise actually present.** v1 chose t* from the nominal σ, but `add_gaussian` clips to [0, 1], so the noise left in the image is weaker (residual std ≈ 0.24–0.26 at σ = 0.3) and the chain over-denoises. Here σ̂ comes from the noisy image alone (`skimage.restoration.estimate_sigma`, wavelet MAD). A Tweedie sweep over t = t_for_sigma(f·σ), f ∈ {0.6, …, 1.15}, plus σ̂ and the residual std, shows how sensitive the result is and where the optimum lies.
2. **Posterior mean from K reverse samples.** One-step Tweedie was well short of the posterior mean at σ ≥ 0.2 (reverse/Tweedie MSE ratio 1.41 and 1.09, against 2 for exact quantities). K = 8 independent reverse chains from the same start give running means for K = 1, 2, 4, 8; extrapolating MSE_K = a + b/K estimates the sampler's posterior-mean error a.

For a fair comparison, DRUNet and BM3D (which were told the nominal σ) are re-run with the same σ̂. Restormer and SCUNet are blind; their scores come from `jepa-denoise-comparison`.

Real camera pairs: σ̂ is unreliable there (spatially correlated noise), so they run under both rules: v1's std(noisy − clean) and σ̂.

Kaggle: GPU T4 ×2 (one ADM copy per GPU, batches split), internet on, outputs of `jepa-guided-diffusion` (v1) and `jepa-denoise-comparison` attached.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "bm3d"], check=True)

In [ ]:
import copy, glob, importlib, io, json, math, os, random, re, subprocess, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.restoration import estimate_sigma

# ---- knobs ----
K = 8                                  # reverse samples per case
SWEEP = [0.6, 0.7, 0.8, 0.9, 1.0, 1.15]  # Tweedie start step t_for_sigma(f * sigma)
N_EVAL, SIGMAS = 16, [0.1, 0.2, 0.3]
SAVE_IMAGES = [0, 1, 2, 3, 9]          # DIV2K cases whose outputs are saved as PNG; the real pairs are always saved

assert torch.cuda.is_available(), "GPU required"
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
D0 = DEVICES[0]
torch.backends.cudnn.benchmark = True
print([torch.cuda.get_device_name(i) for i in range(len(DEVICES))], torch.__version__)
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/adm_cache"), "outputs"
INPUT_ROOTS = ["/kaggle/input", os.environ.get("INPUT_DIR", "/workspace/inputs")]
for d in (CACHE, OUT, f"{OUT}/images"):
    os.makedirs(d, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
S = 256
TIMES = {}


def find_input(pattern):
    """First file matching the glob pattern under the input roots (Kaggle-attached outputs or $INPUT_DIR), else None."""
    for root in INPUT_ROOTS:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return hits[0]
    return None


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if size and have == size:
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def load_zip_images(path, short):
    with zipfile.ZipFile(path) as z:
        names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))

    def one(name):
        with zipfile.ZipFile(path) as z:
            im = Image.open(io.BytesIO(z.read(name))).convert("RGB")
        return torch.from_numpy(np.asarray(TF.resize(im, short)).copy()).permute(2, 0, 1).contiguous()

    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(one, names))


def fetch_image(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")


def add_gaussian(x, sigma, seed=0):
    return (x + sigma * torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))).clamp(0, 1)


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def save_png(img, path):
    Image.fromarray((img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)).save(path)


V1_SCORES = find_input("jepa-guided-diffusion/**/scores.csv")
assert V1_SCORES, "attach the output of the jepa-guided-diffusion notebook (version 1) as input data"
ORIGINAL = pd.read_csv(V1_SCORES)

## 1 · Inputs and noise levels
v1's cases, rebuilt and checked against its noisy-input PSNR. `sigma_rule` is v1's choice (nominal σ; std(noisy − clean) for the real pairs), `sigma_residual` the noise actually left after clipping, `sigma_hat` the blind estimate from the noisy image.

In [ ]:
t0 = time.time()
DIV2K = "https://data.vision.ee.ethz.ch/cvl/DIV2K/"
VALID_IMGS = load_zip_images(download(DIV2K + "DIV2K_valid_HR.zip", f"{CACHE}/DIV2K_valid_HR.zip", size=448_993_893), S)
EVAL_CLEAN = torch.stack([TF.center_crop(im, S).float() / 255 for im in VALID_IMGS[:N_EVAL]])
POLYU = "https://raw.githubusercontent.com/csjunxu/PolyU-Real-World-Noisy-Images-Dataset/master/CroppedImages/"
REAL = []
for n in ["Canon80D_8_8_3200_ball_1", "Canon5D2_5_160_6400_bicycle_10", "NikonD800_5_100_4000_flower_15", "Sony_4-5_125_3200_plant_13"]:
    noisy, clean = (TF.to_tensor(TF.center_crop(fetch_image(f"{POLYU}{n}_{k}.JPG"), S)) for k in ("real", "mean"))
    REAL.append((n, noisy, clean))
CASES = [dict(image=f"div2k_{i}", noise=f"gaussian {s}", sigma=s, clean=y, noisy=add_gaussian(y, s, seed=1000 * i + j))
         for i, y in enumerate(EVAL_CLEAN) for j, s in enumerate(SIGMAS)]
CASES += [dict(image=n, noise="real camera", sigma=float((noisy - clean).std()), clean=clean, noisy=noisy) for n, noisy, clean in REAL]
SAVE_KEYS = {f"div2k_{i}" for i in SAVE_IMAGES}
for k, case in enumerate(CASES):
    case["seed"] = k
    case["save"] = case["noise"] == "real camera" or case["image"] in SAVE_KEYS
    case["sigma_rule"] = case["sigma"]
    case["sigma_residual"] = float((case["noisy"] - case["clean"]).std())
    case["sigma_hat"] = float(estimate_sigma(case["noisy"].permute(1, 2, 0).numpy(), channel_axis=-1, average_sigmas=True))

noisy_psnr = ORIGINAL[ORIGINAL.method == "noisy input"].set_index(["image", "noise"]).psnr
INPUT_GAP = max(abs(peak_signal_noise_ratio(c["clean"].permute(1, 2, 0).numpy(), c["noisy"].permute(1, 2, 0).numpy(), data_range=1.0)
                    - noisy_psnr[(c["image"], c["noise"])]) for c in CASES)
print(f"{len(CASES)} cases; max |noisy-input PSNR - v1| = {INPUT_GAP:.1e} dB")
assert INPUT_GAP < 1e-3, "the inputs differ from the guided-diffusion run"
LEVELS = pd.DataFrame([{k: c[k] for k in ("image", "noise", "sigma_rule", "sigma_residual", "sigma_hat")} for c in CASES])
LEVELS.to_csv(f"{OUT}/noise_levels.csv", index=False)
print(LEVELS.groupby("noise", sort=False)[["sigma_rule", "sigma_residual", "sigma_hat"]].agg(["mean", "min", "max"]).round(4).to_string())
TIMES["inputs"] = time.time() - t0

## 2 · Frozen ADM on every GPU
Same model, checkpoint and fp16 conversion as v1; ε is the first three output channels of ADM's own forward (v1 asserted its re-implemented forward equals this).

In [ ]:
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model

adm = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                   attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                   use_scale_shift_norm=True)  # the README's flags for 256x256_diffusion_uncond.pt
ADM_CKPT = download("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt",
                    f"{CACHE}/256x256_diffusion_uncond.pt", size=2_211_383_297)
adm.load_state_dict(torch.load(ADM_CKPT, map_location="cpu", weights_only=True))
adm.to(D0).eval().requires_grad_(False).convert_to_fp16()
ADMS = [adm] + [copy.deepcopy(adm).to(d) for d in DEVICES[1:]]
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)  # ADM's linear schedule
ABAR = torch.cumprod(1 - BETAS, 0)


def t_for_sigma(sigma):  # pixel σ on [0, 1] -> step whose noise level matches (x in [-1, 1] doubles σ)
    return int((ABAR - 1 / (1 + (2 * sigma) ** 2)).abs().argmin())


def to01(z):
    return ((z + 1) / 2).clamp(0, 1).cpu()


@torch.no_grad()
def eps_model(x, t):
    """ADM's ε prediction for x [B,3,S,S] on D0 and timesteps t [B]; the batch is split across the GPUs."""
    parts = [(m, xi, ti) for m, xi, ti in zip(ADMS, x.chunk(len(ADMS)), t.chunk(len(ADMS))) if len(xi)]
    outs = [m(xi.to(next(m.parameters()).device), ti.to(next(m.parameters()).device)) for m, xi, ti in parts]
    return torch.cat([o[:, :3].float().to(D0) for o in outs])


def kept_fraction(out, case):
    noise = case["noisy"] - case["clean"]
    return float(((out - case["clean"]) * noise).sum() / (noise * noise).sum())


import lpips

LPIPS = lpips.LPIPS(net="alex", verbose=False).to(D0)


def scores(out, clean):
    a, b = out.permute(1, 2, 0).numpy(), clean.permute(1, 2, 0).numpy()
    return dict(psnr=peak_signal_noise_ratio(b, a, data_range=1.0),
                ssim=structural_similarity(b, a, channel_axis=-1, data_range=1.0),
                lpips=LPIPS(out[None].to(D0) * 2 - 1, clean[None].to(D0) * 2 - 1).item())

## 3 · Tweedie start-step sweep
One forward pass per start step: the one-step posterior-mean estimate at t = t_for_sigma(f·σ_rule), at t_for_sigma(σ̂) and at t_for_sigma(σ_residual).

In [ ]:
@torch.no_grad()
def tweedie_at(y, steps):
    """One-step posterior-mean estimates of one noisy image [3,S,S] started at each step in `steps`."""
    t = torch.tensor(steps, device=D0)
    a = ABAR[t.cpu()].float().view(-1, 1, 1, 1).to(D0)
    x = a.sqrt() * (2 * y.to(D0) - 1)[None]
    x0 = ((x - (1 - a).sqrt() * eps_model(x, t)) / a.sqrt()).clamp(-1, 1)
    return to01(x0)


t0, sweep_rows, KEEP = time.time(), [], {}
for case in CASES:
    rules = {f"{f:g}×σ": f * case["sigma_rule"] for f in SWEEP}
    rules.update({"σ̂": case["sigma_hat"], "residual σ": case["sigma_residual"]})
    steps = {name: t_for_sigma(s) for name, s in rules.items()}
    unique = sorted(set(steps.values()))
    outs = dict(zip(unique, tweedie_at(case["noisy"], unique)))
    for name, step in steps.items():
        sweep_rows.append(dict(image=case["image"], noise=case["noise"], rule=name, sigma_used=rules[name], t=step,
                               kept=kept_fraction(outs[step], case), **scores(outs[step], case["clean"])))
    if case["save"]:
        KEEP[(case["image"], case["noise"])] = {"clean (reference)": case["clean"], "noisy input": case["noisy"],
                                                "ADM Tweedie": outs[steps["1×σ"]], "ADM Tweedie (σ̂)": outs[steps["σ̂"]]}
sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(f"{OUT}/tweedie_sweep.csv", index=False)
TIMES["tweedie sweep"] = time.time() - t0
order = [f"{f:g}×σ" for f in SWEEP] + ["σ̂", "residual σ"]
print("mean Tweedie PSNR by start rule")
print(sweep.pivot_table(index="rule", columns="noise", values="psnr").reindex(order).round(3).to_string())
print("\nmean start step t by rule")
print(sweep.pivot_table(index="rule", columns="noise", values="t").reindex(order).round(1).to_string())

## 4 · Mean of K reverse samples
K independent reverse chains from the same start x_t* (one batch, different noise draws), started at t*(σ̂); the real pairs also at v1's t*. Running means give K = 1, 2, 4, 8.

In [ ]:
@torch.no_grad()
def reverse_samples(y, sigma, k, seed):
    """k independent posterior samples by v1's reverse process from t* = t_for_sigma(sigma)."""
    t0 = t_for_sigma(sigma)
    gen = torch.Generator(device=D0).manual_seed(seed)
    x = (ABAR[t0].sqrt().float() * (2 * y.to(D0) - 1))[None].repeat(k, 1, 1, 1)
    for t in range(t0, -1, -1):
        a_t, beta_t = ABAR[t].item(), BETAS[t].item()
        a_prev = ABAR[t - 1].item() if t > 0 else 1.0
        eps = eps_model(x, torch.full((k,), t, device=D0, dtype=torch.long))
        x0 = ((x - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5).clamp(-1, 1)
        if t == 0:
            x = x0
            break
        mean = (a_prev ** 0.5 * beta_t / (1 - a_t)) * x0 + ((1 - beta_t) ** 0.5 * (1 - a_prev) / (1 - a_t)) * x
        x = mean + (beta_t * (1 - a_prev) / (1 - a_t)) ** 0.5 * torch.randn((k, 3, S, S), generator=gen, device=D0)
    return to01(x)


ARM_RULES = {"σ̂": "sigma_hat", "v1 rule": "sigma_rule"}
t0, rows, kmse_rows = time.time(), [], []
for n, case in enumerate(CASES):
    arms = ["σ̂"] + (["v1 rule"] if case["noise"] == "real camera" else [])
    for arm in arms:
        samples = reverse_samples(case["noisy"], case[ARM_RULES[arm]], K, seed=1000 * case["seed"] + 7)
        running = samples.cumsum(0) / torch.arange(1, K + 1).view(-1, 1, 1, 1)
        for k in range(1, K + 1):
            kmse_rows.append(dict(image=case["image"], noise=case["noise"], arm=arm, k=k,
                                  mse=float((running[k - 1] - case["clean"]).pow(2).mean())))
        for k in (1, 2, 4, 8):
            if k > K:
                continue
            out = samples[0] if k == 1 else running[k - 1]
            method = f"ADM reverse, 1 sample ({arm})" if k == 1 else f"ADM mean of {k} ({arm})"
            rows.append(dict(image=case["image"], noise=case["noise"], method=method, kept=kept_fraction(out, case), **scores(out, case["clean"])))
            if case["save"] and arm == "σ̂" and k in (1, K):
                KEEP[(case["image"], case["noise"])][method] = out
    if n % 6 == 0 or n == len(CASES) - 1:
        print(f"  case {n + 1}/{len(CASES)}  {(time.time() - t0) / 60:.1f} min", flush=True)
kmse = pd.DataFrame(kmse_rows)
kmse.to_csv(f"{OUT}/mse_by_k.csv", index=False)
TIMES["reverse samples"] = time.time() - t0

## 5 · DRUNet and BM3D told σ̂
Same pinned DRUNet weights and BM3D call as `jepa-denoise-comparison`; only the noise level changes from nominal to σ̂.

In [ ]:
import bm3d

KAIR = ("cszn/KAIR", "fc1732f4a4514e42ce15e5b3a1e18c828af47a1e")
for path in ("models/basicblock.py", "models/network_unet.py"):
    dest = f"{CACHE}/KAIR/{path}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    download(f"https://raw.githubusercontent.com/{KAIR[0]}/{KAIR[1]}/{path}", dest)
DRUNET_W = download("https://github.com/cszn/KAIR/releases/download/v1.0/drunet_color.pth", f"{CACHE}/drunet_color.pth", size=130_579_305)
sys.path.insert(0, f"{CACHE}/KAIR")
from models.network_unet import UNetRes  # needs models.basicblock from the same checkout
sys.path.remove(f"{CACHE}/KAIR")
for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
    del sys.modules[name]
DRUNET = UNetRes(in_nc=4, out_nc=3, nc=[64, 128, 256, 512], nb=4, act_mode="R", downsample_mode="strideconv",
                 upsample_mode="convtranspose", bias=False)  # the released DPIR weights have no biases
DRUNET.load_state_dict(torch.load(DRUNET_W, map_location="cpu", weights_only=True), strict=True)
DRUNET.to(D0).eval().requires_grad_(False)

t0 = time.time()
for case in CASES:
    s = case["sigma_hat"]
    with torch.no_grad():
        x = case["noisy"][None].to(D0)
        out = DRUNET(torch.cat([x, torch.full_like(x[:, :1], s)], 1))[0].float().clamp(0, 1).cpu()
    rows.append(dict(image=case["image"], noise=case["noise"], method="DRUNet (σ̂)", kept=kept_fraction(out, case), **scores(out, case["clean"])))
    if case["save"]:
        KEEP[(case["image"], case["noise"])]["DRUNet (σ̂)"] = out
    rgb = case["noisy"].permute(1, 2, 0).numpy().astype(np.float64)
    out = torch.from_numpy(np.ascontiguousarray(bm3d.bm3d_rgb(rgb, sigma_psd=s))).permute(2, 0, 1).float().clamp(0, 1)
    rows.append(dict(image=case["image"], noise=case["noise"], method="BM3D (σ̂)", kept=kept_fraction(out, case), **scores(out, case["clean"])))
    if case["save"]:
        KEEP[(case["image"], case["noise"])]["BM3D (σ̂)"] = out
TIMES["DRUNet and BM3D"] = time.time() - t0
new_scores = pd.DataFrame(rows)
new_scores.to_csv(f"{OUT}/scores.csv", index=False)

## 6 · Results

In [ ]:
tw = sweep[sweep.rule.isin(["1×σ", "σ̂", "residual σ"])].replace({"rule": {"1×σ": "ADM Tweedie (v1 rule, re-run)", "σ̂": "ADM Tweedie (σ̂)", "residual σ": "ADM Tweedie (residual σ, oracle)"}})
tw = tw.rename(columns={"rule": "method"})[["image", "noise", "method", "psnr", "ssim", "lpips"]]
parts = [ORIGINAL, tw, new_scores.drop(columns="kept")]
standard_path = find_input("jepa-denoise-comparison/**/standard_denoisers_scores.csv")
if standard_path:
    parts.append(pd.read_csv(standard_path))
ALL = pd.concat(parts, ignore_index=True)
ALL.to_csv(f"{OUT}/all_scores.csv", index=False)
SHOW = ["noisy input", "ADM Tweedie", "ADM Tweedie (v1 rule, re-run)", "ADM Tweedie (σ̂)", "ADM Tweedie (residual σ, oracle)", "ADM reverse",
        "ADM reverse, 1 sample (σ̂)", "ADM mean of 2 (σ̂)", "ADM mean of 4 (σ̂)", "ADM mean of 8 (σ̂)",
        "ADM reverse, 1 sample (v1 rule)", "ADM mean of 8 (v1 rule)", "BM3D", "BM3D (σ̂)", "DRUNet", "DRUNet (σ̂)", "Restormer", "SCUNet"]
SHOW = [m for m in SHOW if m in set(ALL.method)]
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()} (" + ("lower" if metric == "lpips" else "higher") + " = better)")
    print(ALL.pivot_table(index="method", columns="noise", values=metric).reindex(SHOW).round(3).to_string())
rerun = ALL[ALL.method.isin(["ADM Tweedie", "ADM Tweedie (v1 rule, re-run)"])].pivot_table(index=["image", "noise"], columns="method", values="psnr")
RERUN_GAP = float((rerun["ADM Tweedie"] - rerun["ADM Tweedie (v1 rule, re-run)"]).abs().max())
print(f"\nADM Tweedie re-run vs v1: max |Δ PSNR| {RERUN_GAP:.4f} dB")

FIT = {}
for (noise, arm), g in kmse.groupby(["noise", "arm"]):
    m = g.groupby("k").mse.mean()
    b, a = np.polyfit(1.0 / m.index.to_numpy(), m.to_numpy(), 1)   # mse_k = a + b / k
    tweedie = sweep[(sweep.noise == noise) & (sweep.rule == ("σ̂" if arm == "σ̂" else "1×σ"))]
    tweedie_mse = float(np.mean([10 ** (-p / 10) for p in tweedie.psnr]))
    FIT[f"{noise} | {arm}"] = dict(mse_1=float(m.iloc[0]), mse_K=float(m.iloc[-1]), mse_inf=float(a), slope=float(b), tweedie_mse=tweedie_mse)
fit = pd.DataFrame(FIT).T
fit["sample/posterior-mean"] = fit.mse_1 / fit.mse_inf
fit["Tweedie/posterior-mean"] = fit.tweedie_mse / fit.mse_inf
print("\nMSE of the mean of k samples, fitted as a + b/k (a: the sampler's posterior-mean error)")
print(fit.round(6).to_string())

## 7 · Figures

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.3))
for noise, g in sweep[sweep.rule.str.endswith("×σ")].groupby("noise", sort=False):
    m = g.groupby("rule").psnr.mean()
    axs[0].plot([float(r[:-2]) for r in m.index], m.to_numpy(), "o-", label=noise)
axs[0].set(xlabel="f (start step t_for_sigma(f · σ_rule))", ylabel="mean Tweedie PSNR, dB", title="Tweedie: start-step sweep")
axs[0].legend(fontsize=8)
for noise, g in kmse.groupby("noise", sort=False):
    for arm, h in g.groupby("arm"):
        m = h.groupby("k").mse.mean()
        axs[1].plot(m.index, 10 * np.log10(1 / m.to_numpy()), "o-", label=f"{noise} ({arm})")
axs[1].set(xlabel="K samples averaged", ylabel="PSNR of the mean MSE, dB", title="mean of K reverse samples")
axs[1].legend(fontsize=7)
lv = LEVELS[LEVELS.noise != "real camera"]
axs[2].scatter(lv.sigma_residual, lv.sigma_hat, s=14)
axs[2].plot([0, 0.3], [0, 0.3], "k--", lw=0.8)
axs[2].set(xlabel="noise actually present (residual std)", ylabel="σ̂ from the noisy image", title="blind noise estimate")
plt.tight_layout(); plt.savefig(f"{OUT}/sweep_and_k.png", dpi=110); plt.show()

shown = ALL.set_index(["image", "noise", "method"])
for noise in ("gaussian 0.3", "gaussian 0.2", "real camera"):
    keys = [key for key in KEEP if key[1] == noise]
    if not keys:
        continue
    names = list(KEEP[keys[0]])
    fig, axs = plt.subplots(len(keys), len(names), figsize=(2.3 * len(names), 2.45 * len(keys)), squeeze=False)
    for r, key in enumerate(keys):
        for j, name in enumerate(names):
            img = KEEP[key][name]
            title = name if name == "clean (reference)" or (*key, name) not in shown.index else \
                f"{name}\n{shown.loc[(*key, name)].psnr:.2f} dB | LPIPS {shown.loc[(*key, name)].lpips:.3f}"
            axs[r, j].imshow(img.permute(1, 2, 0).numpy()); axs[r, j].set_title(title, fontsize=7)
            axs[r, j].set_xticks([]); axs[r, j].set_yticks([])
        axs[r, 0].set_ylabel(key[0][:14], fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT}/grid_{slug(noise)}.png", dpi=100); plt.close(fig)
for key, panels in KEEP.items():
    folder = f"{OUT}/images/{slug(key[0])}__{slug(key[1])}"
    os.makedirs(folder, exist_ok=True)
    for j, (name, img) in enumerate(panels.items()):
        save_png(img, f"{folder}/{j:02d}-{slug(name)}.png")

with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(K=K, sweep=SWEEP, input_psnr_gap_db=INPUT_GAP, tweedie_rerun_gap_db=RERUN_GAP, kair=KAIR[1],
                   minutes={k: v / 60 for k, v in TIMES.items()}, fit=FIT,
                   mean=ALL.groupby(["noise", "method"], sort=False)[["psnr", "ssim", "lpips"]].mean().reset_index().to_dict("records")),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)))